<a href="https://colab.research.google.com/github/viche09cpsn/retail-sales-analysis/blob/development/retail_sales_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Python para Ciencia de Datos / Proyecto I. Análisis y predicción de ventas-Part 1 (Core)

# PROYECTO: ANÁLISIS DE VENTAS DE UNA TIENDA
# Parte 1: análisis básico con NumPy.
#
# INSTRUCCIONES PARA GOOGLE COLAB:
# 1. Descarga el CSV desde Kaggle:
#    https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset
# 2. Si se descarga un ZIP, descomprímelo.
# 3. Ejecuta este código y selecciona el archivo CSV.
#
# CONFIGURACIÓN EN GITHUB:
# 1. Crea un repositorio llamado retail-sales-analysis con un README.md.
# 2. Conserva la rama main.
# 3. Desde el selector de ramas, crea una rama llamada development.
# 4. Guarda tu cuaderno de Colab en development desde:
#    Archivo > Guardar una copia en GitHub.
#
# Esta parte analiza ventas históricas. Todavía no realiza predicciones.

import csv
import io
import numpy as np
from google.colab import files


# --------------------------------------------------
# 1. CARGAR EL ARCHIVO CSV
# --------------------------------------------------

def cargar_datos():
    print("Selecciona el archivo CSV de ventas.")
    archivos = files.upload()

    if len(archivos) != 1:
        raise ValueError("Debes seleccionar un solo archivo CSV.")

    nombre = list(archivos.keys())[0]

    if not nombre.lower().endswith(".csv"):
        raise ValueError("El archivo seleccionado debe tener extensión .csv.")

    contenido = archivos[nombre].decode("utf-8-sig")

    # Leer los encabezados permite revisar las columnas del archivo.
    encabezados = next(csv.reader(io.StringIO(contenido)), [])
    encabezados = [columna.strip() for columna in encabezados]

    columnas_necesarias = [
        "Transaction ID",
        "Date",
        "Product Category",
        "Quantity",
        "Price per Unit",
        "Total Amount"
    ]

    for columna in columnas_necesarias:
        if columna not in encabezados:
            raise ValueError(f"Falta la columna: {columna}")

    # dtype=str conserva tanto los textos como los números.
    # Los números se convertirán después de revisar cada fila.
    datos = np.genfromtxt(
        io.StringIO(contenido),
        delimiter=",",
        skip_header=1,
        dtype=str,
        autostrip=True,
        ndmin=2
    )

    if datos.size == 0:
        raise ValueError("El archivo no contiene datos.")

    if datos.shape[1] != len(encabezados):
        raise ValueError("La cantidad de columnas no coincide con el encabezado.")

    print("\nArchivo cargado:", nombre)
    print("Filas originales:", len(datos))

    return datos, encabezados


# --------------------------------------------------
# 2. LIMPIAR Y PREPARAR LOS DATOS
# --------------------------------------------------

def limpiar_datos(datos, encabezados):
    # Buscar la posición de cada columna por su nombre.
    columna_id = encabezados.index("Transaction ID")
    columna_fecha = encabezados.index("Date")
    columna_categoria = encabezados.index("Product Category")
    columna_cantidad = encabezados.index("Quantity")
    columna_precio = encabezados.index("Price per Unit")
    columna_total = encabezados.index("Total Amount")

    fechas = []
    categorias = []
    cantidades = []
    precios = []
    ventas = []

    identificadores_vistos = set()
    filas_invalidas = 0
    filas_duplicadas = 0
    totales_incorrectos = 0

    for fila in datos:
        identificador = fila[columna_id].strip()
        categoria = fila[columna_categoria].strip()
        texto_fecha = fila[columna_fecha].strip()

        # No podemos analizar una venta sin identificador o categoría.
        if identificador == "" or categoria == "":
            filas_invalidas += 1
            continue

        try:
            fecha = np.datetime64(texto_fecha, "D")
            cantidad = float(fila[columna_cantidad])
            precio = float(fila[columna_precio])
            total = float(fila[columna_total])

            numeros = np.array([cantidad, precio, total])

            if np.isnat(fecha) or not np.all(np.isfinite(numeros)):
                filas_invalidas += 1
                continue

            # Las unidades deben ser enteras y positivas.
            # Para este ejercicio no se analizan devoluciones.
            if cantidad <= 0 or cantidad != int(cantidad):
                filas_invalidas += 1
                continue

            if precio < 0 or total < 0:
                filas_invalidas += 1
                continue

        except (ValueError, OverflowError):
            filas_invalidas += 1
            continue

        # Revisar que el importe coincida con cantidad × precio.
        if not np.isclose(total, cantidad * precio, rtol=0, atol=0.01):
            totales_incorrectos += 1
            continue

        # Cada Transaction ID debe representar una sola venta.
        if identificador in identificadores_vistos:
            filas_duplicadas += 1
            continue

        identificadores_vistos.add(identificador)

        fechas.append(fecha)
        categorias.append(categoria)
        cantidades.append(int(cantidad))
        precios.append(precio)
        ventas.append(total)

    if len(ventas) == 0:
        raise ValueError("No quedaron ventas válidas después de la limpieza.")

    print("\nRESUMEN DE LIMPIEZA")
    print("Filas inválidas descartadas:", filas_invalidas)
    print("Identificadores repetidos descartados:", filas_duplicadas)
    print("Importes que no coinciden con cantidad × precio:", totales_incorrectos)
    print("Filas válidas:", len(ventas))

    return (
        np.array(fechas, dtype="datetime64[D]"),
        np.array(categorias),
        np.array(cantidades, dtype=int),
        np.array(precios, dtype=float),
        np.array(ventas, dtype=float)
    )


# --------------------------------------------------
# 3. ANALIZAR LAS VENTAS POR CATEGORÍA
# --------------------------------------------------

def analizar_categorias(fechas, categorias, ventas):
    categorias_unicas = np.unique(categorias)

    fecha_inicio = np.min(fechas)
    fecha_fin = np.max(fechas)

    # Crear todos los días del período, incluyendo los que no tienen ventas.
    dias = np.arange(
        fecha_inicio,
        fecha_fin + np.timedelta64(1, "D"),
        np.timedelta64(1, "D")
    )

    totales = []
    promedios = []

    print("\nVENTAS POR CATEGORÍA")
    print("Período:", fecha_inicio, "hasta", fecha_fin)
    print("Cantidad de días del período:", len(dias))
    print("Los promedios incluyen días sin ventas como cero.")

    for categoria in categorias_unicas:
        filtro = categorias == categoria

        fechas_categoria = fechas[filtro]
        ventas_categoria = ventas[filtro]

        total_categoria = np.sum(ventas_categoria)

        # Guardar cuánto vendió esta categoría en cada día.
        ventas_diarias = np.zeros(len(dias))

        for i in range(len(dias)):
            filtro_dia = fechas_categoria == dias[i]
            ventas_diarias[i] = np.sum(ventas_categoria[filtro_dia])

        promedio_diario = np.mean(ventas_diarias)

        totales.append(total_categoria)
        promedios.append(promedio_diario)

        print(f"\nCategoría: {categoria}")
        print(f"  Total vendido: {total_categoria:,.2f}")
        print(f"  Promedio diario: {promedio_diario:,.2f}")

    totales = np.array(totales)
    promedios = np.array(promedios)

    # Mostrar todas las categorías si hay empate.
    categorias_mayores = categorias_unicas[totales == np.max(totales)]
    categorias_menores = categorias_unicas[totales == np.min(totales)]

    print("\nCATEGORÍAS CON MAYORES Y MENORES VENTAS")
    print(
        "Mayor venta:",
        ", ".join(categorias_mayores),
        f"con {np.max(totales):,.2f}"
    )
    print(
        "Menor venta:",
        ", ".join(categorias_menores),
        f"con {np.min(totales):,.2f}"
    )

    # Comprobar que los totales por categoría sumen el total general.
    assert np.isclose(np.sum(totales), np.sum(ventas))

    return categorias_unicas, totales, promedios


# --------------------------------------------------
# 4. FILTRAR UNA CATEGORÍA
# --------------------------------------------------

def mostrar_categoria(fechas, categorias, cantidades, precios, ventas):
    categorias_disponibles = np.unique(categorias)

    print("\nFILTRAR VENTAS")
    print("Categorías disponibles:", ", ".join(categorias_disponibles))

    categoria_elegida = input("Escribe una categoría: ").strip()

    # Comparar en minúsculas para aceptar, por ejemplo, beauty o Beauty.
    filtro = np.char.lower(categorias) == categoria_elegida.lower()

    if not np.any(filtro):
        print("No se encontró esa categoría. Revisa las opciones disponibles.")
        return

    print("\nVentas de la categoría seleccionada:")

    for fecha, cantidad, precio, total in zip(
        fechas[filtro],
        cantidades[filtro],
        precios[filtro],
        ventas[filtro]
    ):
        print(
            f"Fecha: {fecha} | Unidades: {cantidad} | "
            f"Precio unitario: {precio:,.2f} | Total: {total:,.2f}"
        )

    print("Cantidad de transacciones:", np.sum(filtro))
    print(f"Total de la categoría: {np.sum(ventas[filtro]):,.2f}")


# --------------------------------------------------
# 5. REALIZAR OPERACIONES MATEMÁTICAS
# --------------------------------------------------

def calcular_estadisticas(cantidades, precios, ventas):
    print("\nOPERACIONES MATEMÁTICAS")

    # SUMA: importe total de las ventas.
    total_ventas = np.sum(ventas)
    total_unidades = np.sum(cantidades)

    print(f"\nSuma: total vendido = {total_ventas:,.2f}")
    print(f"Total de unidades vendidas = {total_unidades}")

    # RESTA: diferencia entre la transacción mayor y la menor.
    diferencia = np.max(ventas) - np.min(ventas)

    print(f"\nResta: diferencia entre venta máxima y mínima = {diferencia:,.2f}")

    # MULTIPLICACIÓN: calcular el importe de cada transacción.
    importes_calculados = cantidades * precios

    print("\nMultiplicación: cantidad × precio, primeras cinco ventas:")
    print(importes_calculados[:5])

    # Escenario de práctica: aplicar un descuento del 10 %.
    ventas_con_descuento = ventas * 0.90

    print(f"Total si se aplicara un descuento del 10 %: {np.sum(ventas_con_descuento):,.2f}")
    print("Este descuento es un ejemplo, no una venta real del archivo.")

    # DIVISIÓN: importe promedio por transacción.
    promedio_transaccion = total_ventas / len(ventas)

    # Precio promedio por unidad, considerando cuántas unidades se vendieron.
    promedio_por_unidad = total_ventas / total_unidades

    print(f"\nDivisión: venta promedio por transacción = {promedio_transaccion:,.2f}")
    print(f"Importe promedio por unidad vendida = {promedio_por_unidad:,.2f}")

    assert np.allclose(importes_calculados, ventas, rtol=0, atol=0.01)


# --------------------------------------------------
# 6. CREAR ARCHIVOS PARA EL PROYECTO
# --------------------------------------------------

def crear_archivos_proyecto(categorias, totales, promedios):
    # Guardar los resultados en un nuevo CSV.
    with open("resumen_ventas.csv", "w", newline="", encoding="utf-8") as archivo:
        escritor = csv.writer(archivo)
        escritor.writerow([
            "Categoria",
            "Total de ventas",
            "Promedio diario de ventas"
        ])

        for categoria, total, promedio in zip(categorias, totales, promedios):
            escritor.writerow([categoria, f"{total:.2f}", f"{promedio:.2f}"])

    readme = """# Retail Sales Analysis

Proyecto de análisis de ventas de una tienda de retail.

## Parte 1

Se utiliza NumPy para:
- Cargar y limpiar el CSV.
- Calcular ventas totales y promedios diarios por categoría.
- Identificar categorías con mayores y menores ventas.
- Filtrar transacciones de una categoría.
- Realizar operaciones matemáticas.

Esta etapa no incluye modelos de predicción.

## Dataset

Retail Sales Dataset:
https://www.kaggle.com/datasets/mohammadtalib786/retail-sales-dataset

Los importes se presentan sin asignar una moneda.

## Instalación y uso

1. Abre el cuaderno en Google Colab.
2. Ejecuta el código y sube el archivo CSV descargado de Kaggle.
3. Escribe una categoría cuando el programa lo solicite.

Google Colab incluye NumPy. La carga interactiva utiliza google.colab,
por lo que este código está preparado para ejecutarse en Colab.

## Limpieza

Se descartan filas con datos obligatorios inválidos, unidades no enteras
o no positivas, importes negativos y totales que no coinciden con
cantidad por precio. Se conserva una venta válida por Transaction ID.

## Promedio diario

Se suman las ventas de cada categoría por fecha y se calcula el promedio
sobre todos los días del período válido, incluyendo días sin ventas.

## Estructura sugerida

- data/: archivo CSV original.
- notebooks/: cuaderno de Google Colab.
- README.md: descripción e instrucciones.
- requirements.txt: dependencias.
- resumen_ventas.csv: resultados del análisis.

## Ramas

- main: versión principal.
- development: desarrollo del análisis.
"""

    with open("README.md", "w", encoding="utf-8") as archivo:
        archivo.write(readme)

    with open("requirements.txt", "w", encoding="utf-8") as archivo:
        archivo.write("numpy\n")

    print("\nArchivos creados en el panel de archivos de Colab:")
    print("- resumen_ventas.csv")
    print("- README.md")
    print("- requirements.txt")
    print("Puedes descargarlos desde ese panel y agregarlos a tu repositorio.")


# --------------------------------------------------
# 7. EJECUTAR EL PROGRAMA
# --------------------------------------------------

try:
    datos, encabezados = cargar_datos()

    fechas, categorias, cantidades, precios, ventas = limpiar_datos(
        datos,
        encabezados
    )

    categorias_unicas, totales, promedios = analizar_categorias(
        fechas,
        categorias,
        ventas
    )

    mostrar_categoria(fechas, categorias, cantidades, precios, ventas)

    calcular_estadisticas(cantidades, precios, ventas)

    crear_archivos_proyecto(categorias_unicas, totales, promedios)

    print("\nAnálisis finalizado.")

except (ValueError, UnicodeDecodeError, csv.Error, OSError) as error:
    print("\nNo se pudo completar el análisis:", error)


Selecciona el archivo CSV de ventas.


Saving retail_sales_dataset.csv to retail_sales_dataset.csv

Archivo cargado: retail_sales_dataset.csv
Filas originales: 1000

RESUMEN DE LIMPIEZA
Filas inválidas descartadas: 0
Identificadores repetidos descartados: 0
Importes que no coinciden con cantidad × precio: 0
Filas válidas: 1000

VENTAS POR CATEGORÍA
Período: 2023-01-01 hasta 2024-01-01
Cantidad de días del período: 366
Los promedios incluyen días sin ventas como cero.

Categoría: Beauty
  Total vendido: 143,515.00
  Promedio diario: 392.12

Categoría: Clothing
  Total vendido: 155,580.00
  Promedio diario: 425.08

Categoría: Electronics
  Total vendido: 156,905.00
  Promedio diario: 428.70

CATEGORÍAS CON MAYORES Y MENORES VENTAS
Mayor venta: Electronics con 156,905.00
Menor venta: Beauty con 143,515.00

FILTRAR VENTAS
Categorías disponibles: Beauty, Clothing, Electronics
Escribe una categoría: Electronics

Ventas de la categoría seleccionada:
Fecha: 2023-01-13 | Unidades: 1 | Precio unitario: 30.00 | Total: 30.00
Fecha: 202